# 5 · Agente con memoria

Lección 7 del módulo 1: [Agente con memoria](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239417-lesson-7-agent-with-memory). El notebook original es [`agent-memory.ipynb`](../../langchain-academy/module-1/agent-memory.ipynb).

El agente del notebook 4 olvida todo apenas termina: cada `invoke` empieza con un estado nuevo. Aquí se le agrega memoria con un **checkpointer**, que guarda el estado después de cada paso, y con **hilos**, que separan una conversación de otra.

## El modelo

Estos apuntes usan un modelo local con [Ollama](https://ollama.com), gratis y sin claves: `qwen2.5:7b`, que sabe llamar herramientas. Antes de ejecutar, con Ollama abierto:

```
ollama pull qwen2.5:7b
```

`temperature=0` hace que el modelo elija siempre la opción más probable, así las respuestas casi no cambian de una ejecución a otra.

El curso usa `gpt-4o` de OpenAI. Para usarlo, cambia la celda de abajo por:

```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o", temperature=0)
```

El resto del notebook queda igual: al grafo no le importa qué modelo hay detrás.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", temperature=0)

## El agente del notebook 4

Las mismas herramientas, el mismo nodo y el mismo grafo:

In [ ]:
def sumar(a: int, b: int) -> int:
    """Suma a y b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a + b


def multiplicar(a: int, b: int) -> int:
    """Multiplica a por b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a * b


def dividir(a: int, b: int) -> float:
    """Divide a por b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a / b


herramientas = [sumar, multiplicar, dividir]
llm_con_herramientas = llm.bind_tools(herramientas)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import MessagesState

mensaje_sistema = SystemMessage(content="Eres un asistente que hace operaciones aritméticas con los datos que te dan.")


def asistente(estado: MessagesState):
    return {"messages": [llm_con_herramientas.invoke([mensaje_sistema] + estado["messages"])]}

In [ ]:
from IPython.display import Image, display
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

constructor = StateGraph(MessagesState)
constructor.add_node("asistente", asistente)
constructor.add_node("tools", ToolNode(herramientas))

constructor.add_edge(START, "asistente")
constructor.add_conditional_edges("asistente", tools_condition)
constructor.add_edge("tools", "asistente")
grafo_react = constructor.compile()

display(Image(grafo_react.get_graph(xray=True).draw_mermaid_png()))

## Sin memoria

In [ ]:
resultado = grafo_react.invoke({"messages": [HumanMessage(content="Suma 3 y 4.")]})
for m in resultado["messages"]:
    m.pretty_print()

Ahora se le pide seguir con el resultado anterior:

In [ ]:
resultado = grafo_react.invoke({"messages": [HumanMessage(content="Multiplica eso por 2.")]})
for m in resultado["messages"]:
    m.pretty_print()

El modelo no sabe qué es «eso»: el estado solo vive mientras dura un `invoke`, y la segunda llamada empieza con un único mensaje. El 7 de la primera se perdió.

## Un checkpointer

Un **checkpointer** guarda el estado del grafo después de cada paso, en un **checkpoint**. Cuando se vuelve a ejecutar el grafo, parte desde el último checkpoint en vez de partir de cero.

`InMemorySaver` guarda los checkpoints en la memoria del proceso. En el video se llama `MemorySaver`, que es el nombre anterior de lo mismo y sigue funcionando. Al reiniciar el kernel se pierde todo, y por eso en el módulo 2 se cambia por uno que guarda en SQLite. El checkpointer se pasa al compilar; el grafo no cambia.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memoria = InMemorySaver()
grafo_react_memoria = constructor.compile(checkpointer=memoria)

## Hilos

Con un checkpointer, cada ejecución necesita un **hilo**, identificado por un `thread_id`. Los checkpoints de un hilo forman su historia, y otro `thread_id` es otra conversación, sin nada guardado.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

resultado = grafo_react_memoria.invoke({"messages": [HumanMessage(content="Suma 3 y 4.")]}, config)
for m in resultado["messages"]:
    m.pretty_print()

Con el mismo `thread_id`, el grafo parte desde el último checkpoint del hilo. El mensaje nuevo se agrega a la conversación guardada gracias al reducer `add_messages` del notebook 2:

In [ ]:
resultado = grafo_react_memoria.invoke({"messages": [HumanMessage(content="Multiplica eso por 2.")]}, config)
for m in resultado["messages"]:
    m.pretty_print()

Esta vez el modelo ve toda la conversación y sabe que «eso» es 7. En otro hilo, la misma pregunta vuelve a quedar sin contexto:

In [ ]:
otro_hilo = {"configurable": {"thread_id": "2"}}

resultado = grafo_react_memoria.invoke({"messages": [HumanMessage(content="Multiplica eso por 2.")]}, otro_hilo)
resultado["messages"][-1].pretty_print()

## Qué guardó el checkpointer

`get_state` devuelve el último checkpoint de un hilo. `values` es el estado y `next` dice qué nodo seguiría; está vacío porque el grafo llegó a `END`.

In [ ]:
estado = grafo_react_memoria.get_state(config)
print(len(estado.values["messages"]), "mensajes en el hilo 1")
print("next:", estado.next)

`get_state_history` devuelve todos los checkpoints del hilo, del más nuevo al más viejo. Cada uno es una foto del estado después de un paso:

In [ ]:
historia = list(grafo_react_memoria.get_state_history(config))
print(len(historia), "checkpoints en el hilo 1\n")
for foto in reversed(historia):
    print(f"paso {foto.metadata['step']:>2}: {len(foto.values.get('messages', []))} mensajes, sigue {foto.next}")

Con esa historia, en el módulo 3 se vuelve a un checkpoint anterior, se cambia y se sigue desde ahí: eso es *time travel*.

## Resumen

- Sin checkpointer, el estado solo dura un `invoke`.
- Un **checkpointer** guarda el estado después de cada paso. `InMemorySaver` lo hace en memoria; `MemorySaver` es su nombre anterior.
- Cada conversación es un **hilo** con su `thread_id`. Con el mismo hilo, el grafo sigue desde el último checkpoint.
- `get_state` muestra el último checkpoint y `get_state_history`, todos.

## Ejercicios

1. Abre un hilo nuevo y conversa tres turnos con el agente. ¿Cuántos checkpoints quedaron? ¿Cuántos pasos agrega cada turno?
2. Reinicia el kernel, vuelve a crear `grafo_react_memoria` y pregunta «Multiplica eso por 2» en el hilo 1. ¿Se acuerda? ¿Por qué?
3. Si un hilo durara cien turnos, ¿qué pasaría con la cantidad de mensajes que recibe el modelo en cada llamada? El módulo 2 trata justo eso: recortar y resumir mensajes.